In [ ]:
import os
import random
import datetime
import numpy as np
import scipy.io as spio

def parseAllCovMat(inputCovMat):
    outputMat = 0
    for a,singleArr in enumerate(inputCovMat):
        if len(np.shape(singleArr)) == 3:
            if np.shape(outputMat) == ():
                outputMat = singleArr
            else:
                outputMat = np.concatenate((outputMat,singleArr),axis=0)
    meanCovMat = np.mean(outputMat.astype(float),axis=0)
    lowerTriangleInds = np.tril_indices(np.shape(meanCovMat)[0])
    meanCovMat[lowerTriangleInds] = 0
    return meanCovMat+np.transpose(meanCovMat)

def generateDirectedGraphsSpatialOnly(name):
    spaFile = ''
    smat = spio.loadmat(spaFile)
    sScale = smat.get('scale')[0][0]
    spaConnection = parseAllCovMat(smat.get('allCovMat')[0]).astype(float)/sScale
    return spaConnection

def parseUnitsByRegion(inputArr):
    pfcI = np.nonzero(inputArr-1)[0].flatten()
    hpcI = np.delete(np.arange(0,len(inputArr),1),pfcI,axis=0)
    return hpcI,pfcI

def loadValuesFromFileSpatial(fileOfInterest,justFileName):
    spaConnectionGraph = generateDirectedGraphsSpatialOnly(justFileName)    
    rmat = spio.loadmat(fileOfInterest)
    hpcInds,pfcInds = parseUnitsByRegion(rmat.get('regionIndex').flatten())
    spaSpikeTrain = rmat.get('spaSpike').astype(int)
    connDict = {'spaConnectionAll':spaConnectionGraph,
                'spaConnectionHPC':np.take(np.take(spaConnectionGraph,hpcInds,axis=1),hpcInds,axis=0),
                'spaConnectionPFC':np.take(np.take(spaConnectionGraph,pfcInds,axis=1),pfcInds,axis=0),
               }
    trainDict = {'spaSpikeTrain':spaSpikeTrain,'spaSpikeTrainHPC':np.take(spaSpikeTrain,hpcInds,axis=0),
                 'spaSpikeTrainPFC':np.take(spaSpikeTrain,pfcInds,axis=0),
                }
    return connDict,trainDict

def phi(inVolt,threshold):
    currArr = inVolt/threshold
    return np.where(currArr>1,1,currArr)

#Implementation of the Galves–Löcherbach model for biophysically simulating real spike trains
#doi:10.1007/s10955-013-0733-9
def galvesLocherbachSimulator(unitCount,connGraph,remain,thresh,timeSteps):
    #Unit Count, Connection Graph, and timeSteps are all known
    random.seed(2022) #Pick a RNG seed, does not matter which one, just make sure it consistent across all runs
    outputSimmedSpikeTrain = 0
    sampV = np.random.randint(0,thresh,size=(unitCount,))
    for t in range(timeSteps):
        spikeProb = phi(sampV,thresh)
        randArr = np.random.uniform(0,thresh*3,size=(unitCount,))
        trueSpike = np.reshape(np.where(spikeProb>randArr,1,0),(len(spikeProb),1)).astype(int)
        if np.shape(outputSimmedSpikeTrain) == ():
            outputSimmedSpikeTrain = trueSpike
        else:
            outputSimmedSpikeTrain = np.concatenate((outputSimmedSpikeTrain,trueSpike),axis=1)
        voltSpike = np.where(spikeProb>randArr,0,spikeProb)
        voltSpike = voltSpike*remain
        sampV = voltSpike+np.sum(voltSpike*connGraph)
    return outputSimmedSpikeTrain.astype(int)

def generateStepSizing(inputConnections):
    thresholdTrue = np.shape(inputConnections)[0]*np.mean(inputConnections)
    thresholdRange = np.arange(1.8*thresholdTrue,2.3*thresholdTrue,.1*thresholdTrue)
    remainRange = np.arange(.6,1.1,.1)
    return thresholdRange,remainRange

def paramWalkSpatial(inputRawFile,qFile):
    errorDict = {}
    connections,trueTrains = loadValuesFromFileSpatial(inputRawFile,qFile)
    connectionKeyList = list(connections.keys())
    trueTrainKeyList = list(trueTrains.keys())
    for a in range(len(connectionKeyList)):
        currConnectionKey = connectionKeyList[a]
        currTrueTrainKey = trueTrainKeyList[a]
        currConnectionGraph = connections[currConnectionKey]
        currTrueSpikeTrain = trueTrains[currTrueTrainKey]
        rangeThreshold,rangeRemain = generateStepSizing(currConnectionGraph)
        zeroArr = np.zeros((len(rangeThreshold),len(rangeRemain)))
        for x in range(np.shape(zeroArr)[0]):
            for y in range(np.shape(zeroArr)[1]):
                falseTrain = galvesLocherbachSimulator(np.shape(currConnectionGraph)[0],currConnectionGraph,
                                                       rangeRemain[y],rangeThreshold[x],50000)
                zeroArr[x,y] += np.sum(np.abs(currTrueSpikeTrain[:,:50000]-falseTrain))/(np.shape(falseTrain)[0]*50000)
        errorDict.update({f'{currTrueTrainKey}Error':np.abs(zeroArr-1)})
    return errorDict

def pathWalk(inRawPath,inSavPath):
    useFiles = []
    for aRoot,aDirs,aFiles in os.walk(inRawPath):
        for aFile in aFiles:
            if aFile.endswith('.mat') and aFile == 'Rat3.17_E_1_session.mat':
                print(aFile)
                currRawFile = os.path.join(inRawPath,aFile)
                currSavFile = os.path.join(inSavPath,aFile)
                dictToSave = paramWalkSpatial(currRawFile,aFile)
                saveVar = spio.savemat(currSavFile,dictToSave)
    return 'Done'

testPath = ''
savePath = ''
doneVar = pathWalk(testPath,savePath)
